# 🌳🌡️ Day 13 · Interactions: trees and heat in Tashkent

**#30DayMapChallenge Uzbekistan 2026** · beginner tutorial, no experience needed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NodiraTillayeva/30DayMapChallengeUzbekistan/blob/main/2026/Day%2013%20Interactions/tutorial/day13_trees_and_heat.ipynb)

In this notebook you will:
1. search a **satellite archive** with **STAC**, like a search engine for satellite images,
2. load a **Landsat** image of Tashkent from a summer morning,
3. turn its thermal band into **surface temperature** in °C,
4. measure how green every place is with **NDVI**,
5. see the **interaction**: the more trees, the cooler the ground.

No account needed: the data comes free from the **Microsoft Planetary Computer**.

Run the cells from top to bottom: click a cell, then press **Shift + Enter**.

## Step 0 · Get ready

Three new tools:
- **pystac-client** searches satellite archives,
- **planetary-computer** gives us free access to the Planetary Computer's files,
- **odc-stac** loads the images we found as a grid of numbers.

In [ ]:
!pip -q install pystac-client planetary-computer odc-stac

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pystac_client, planetary_computer, odc.stac
from pyproj import Transformer

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
print("Ready! ✅")

## Step 1 · Search for Landsat images of Tashkent

We ask the archive for **Landsat 8 and 9** images that cover the centre of Tashkent, taken in **summer 2026**, with **less than 5% clouds**. Each image is called an **item**.

In [ ]:
search = catalog.search(
    collections=["landsat-c2-l2"],
    intersects={"type": "Point", "coordinates": [69.27, 41.31]},   # lon, lat of Tashkent
    datetime="2026-06-01/2026-09-30",
    query={"eo:cloud_cover": {"lt": 5}, "platform": {"in": ["landsat-8", "landsat-9"]}},
)
items = list(search.items())

pd.DataFrame([{"id": i.id, "date": i.datetime.date(), "satellite": i.properties["platform"],
               "clouds %": i.properties["eo:cloud_cover"]} for i in items]).sort_values("date")

## Step 2 · Pick one image

We take **28 July 2026**: Landsat 8, almost no clouds, a hot summer day. The satellite passed over Tashkent at about **11:04** local time.

This is a small preview of the whole image, 185 km wide. Tashkent is the grey patch in the middle.

In [ ]:
item = catalog.get_collection("landsat-c2-l2").get_item("LC08_L2SP_154031_20260728_02_T1")
print(item.datetime)

from IPython.display import Image
Image(url=item.assets["rendered_preview"].href, width=500)

## Step 3 · Load Tashkent

We cut out a box around Tashkent and load three **bands** (layers of the image), each pixel 30 m × 30 m:
- `red` and `nir08` (near-infrared): for greenness,
- `lwir11` (thermal infrared): for temperature.

In [ ]:
bbox = [69.10, 41.20, 69.45, 41.42]     # west, south, east, north

ds = odc.stac.load([item], bands=["red", "nir08", "lwir11"], bbox=bbox,
                   crs="EPSG:32642", resolution=30).isel(time=0)
ds

## Step 4 · Surface temperature

The thermal band stores numbers, not degrees. The USGS gives the recipe to turn them into **kelvin**; minus 273.15 gives **°C**.

This is the temperature of the **surface**: roofs, asphalt, soil, tree tops. It's much hotter than the air temperature in the weather forecast.

In [ ]:
temp = ds["lwir11"] * 0.00341802 + 149.0 - 273.15

print("Coolest:", round(float(temp.min()), 1), "°C")
print("Hottest:", round(float(temp.max()), 1), "°C")
print("Average:", round(float(temp.mean()), 1), "°C")

temp.plot.imshow(cmap="inferno", vmin=30, vmax=55, size=7, cbar_kwargs={"label": "surface temperature (°C)"})
plt.title("")
plt.axis("off")
plt.show()

## Step 5 · How green? NDVI

Healthy leaves reflect a lot of **near-infrared** light and absorb **red** light. **NDVI** compares the two:

NDVI = (NIR − red) / (NIR + red)

- around **0**: roofs, roads, bare soil,
- **0.2–0.4**: grass, gardens, sparse trees,
- **0.6** and more: dense trees.

Water is below 0.

In [ ]:
red = ds["red"] * 0.0000275 - 0.2      # turn the numbers into reflectance (0–1)
nir = ds["nir08"] * 0.0000275 - 0.2
ndvi = ((nir - red) / (nir + red)).clip(-1, 1)

ndvi.plot.imshow(cmap="YlGn", vmin=0, vmax=0.8, size=7, cbar_kwargs={"label": "NDVI"})
plt.title("")
plt.axis("off")
plt.show()

## Step 6 · The interaction: greener means cooler

Each pixel has two numbers: its NDVI and its temperature. Let's put **all 800,000 pixels** on one chart. Dark hexagons = many pixels.

In [ ]:
x = ndvi.values.ravel()
y = temp.values.ravel()
land = x > 0                          # leave out water

plt.figure(figsize=(7, 5))
plt.hexbin(x[land], y[land], gridsize=60, cmap="Greys", bins="log")
plt.xlabel("NDVI (greenness)")
plt.ylabel("surface temperature (°C)")
plt.show()

print("Correlation:", round(np.corrcoef(x[land], y[land])[0, 1], 2))

In [ ]:
groups = pd.cut(x[land], [0, 0.2, 0.4, 0.6, 1], labels=["bare (0–0.2)", "a little green (0.2–0.4)",
                                                       "green (0.4–0.6)", "trees (0.6+)"])
pd.Series(y[land]).groupby(groups, observed=True).mean().round(1).rename("average °C")

## Step 7 · Famous places

`Transformer` turns longitude and latitude into the image's metres (UTM zone 42N). Then we take the average of a 90 m × 90 m square around each place.

Add your school, your mahalla or your favourite park!

In [ ]:
places = {
    "Botanical Garden":   (41.3445, 69.3153),
    "Japanese Garden":    (41.3396, 69.2820),
    "Tashkent City Park": (41.3165, 69.2484),
    "Chilonzor metro":    (41.2744, 69.2050),
    "Chorsu Bazaar":      (41.3262, 69.2369),
    "Airport":            (41.2600, 69.2851),
}

to_utm = Transformer.from_crs("EPSG:4326", "EPSG:32642", always_xy=True)
rows = []
for name, (lat, lon) in places.items():
    px, py = to_utm.transform(lon, lat)
    box = dict(x=slice(px - 45, px + 45), y=slice(py + 45, py - 45))
    rows.append({"place": name, "°C": round(float(temp.sel(**box).mean()), 1),
                 "NDVI": round(float(ndvi.sel(**box).mean()), 2)})
pd.DataFrame(rows).sort_values("°C")

## Step 8 · Save your map

Temperature and greenness side by side, without titles, saved as **trees_and_heat.png**.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), facecolor="white")
temp.plot.imshow(ax=ax1, cmap="inferno", vmin=30, vmax=55, add_colorbar=False)
ndvi.plot.imshow(ax=ax2, cmap="YlGn", vmin=0, vmax=0.8, add_colorbar=False)
for ax in (ax1, ax2):
    ax.set_axis_off()
    ax.set_title("")
plt.savefig("trees_and_heat.png", dpi=200, bbox_inches="tight")

from google.colab import files
files.download("trees_and_heat.png")

## 🎉 Done!

Share your map with **#30DayMapChallenge**.

**Credits:** Landsat 8 Collection 2 Level-2 courtesy of the U.S. Geological Survey, accessed through the Microsoft Planetary Computer.